# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
if IS_COLAB or IS_KAGGLE:
    !pip install optuna

import optuna

In [4]:
import importlib
import scipy.sparse as sps
import pandas as pd
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import hyperparameter_tuning

Running on local — storage at: /home/luigi/RecSys
Running on local — storage at: /home/luigi/RecSys


# **Load Data**

In [5]:
# Load datasets
URM_train = sps.load_npz(paths.URM_TRAIN)
URM_validation = sps.load_npz(paths.URM_VALIDATION)

In [6]:
def evaluate_recommender(recommender, at):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Hyperparameter search**

In [7]:
# Define objective function for hyperparameter tuning
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender

STUDY_NAME = EASE_R_Recommender.RECOMMENDER_NAME

def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    recommender_instance = EASE_R_Recommender(URM_train)

    use_topk = optuna_trial.suggest_categorical("use_topk", [False, True])
    topK = None
    if use_topk:
        topK = optuna_trial.suggest_int("topK", 10, 2000, step=10)

    recommender_instance.fit(
        l2_norm=optuna_trial.suggest_float("l2", 1e-6, 1e6, log=True),
        normalize_matrix=optuna_trial.suggest_categorical("normalize", [False, True]),
        topK=topK
    )

    return evaluate_recommender(recommender_instance, at=20)

In [8]:
# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    objective_function,
    study_name=STUDY_NAME,
    n_trials=100
)

[I 2025-11-08 22:16:22,913] A new study created in RDB with name: EASE_R_Recommender


  0%|          | 0/100 [00:00<?, ?it/s]

EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 10.63 sec
[I 2025-11-08 22:16:41,760] Trial 0 finished with value: 0.26322194933891296 and parameters: {'use_topk': False, 'l2': 21.436677916019097, 'normalize': False}. Best is trial 0 with value: 0.26322194933891296.
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 10.65 sec
[I 2025-11-08 22:17:00,962] Trial 1 finished with value: 0.22336189448833466 and parameters: {'use_topk': False, 'l2': 0.2938949451130356, 'normalize': False}. Best is trial 0 with value: 0.26322194933891296.
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 10.76 sec
[I 2025-11-08 22:17:24,132] Trial 2 finished with value: 0.22132951021194458 and parameters: {'use_topk': True, 'topK': 1150, 'l2': 595974.6733727442, 'normalize': True}. Best is trial 0 with value: 0.26322194933891296.
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 

In [9]:
optuna.visualization.plot_optimization_history(optuna_study)

In [10]:
optuna.visualization.plot_param_importances(optuna_study)

In [11]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

# **Hyperparameter tuning**

In [14]:
STUDY_NAME = EASE_R_Recommender.RECOMMENDER_NAME + "_tuning_1"

def tuning_function(optuna_trial: optuna.trial.Trial) -> float:
    recommender_instance = EASE_R_Recommender(URM_train)

    recommender_instance.fit(
        l2_norm=optuna_trial.suggest_float("l2", 50, 3000, log=True),
        normalize_matrix=False,
        topK=optuna_trial.suggest_int("topK", 700, 1000, step=25)
    )

    return evaluate_recommender(recommender_instance, at=20)

In [15]:
# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    tuning_function,
    study_name=STUDY_NAME,
    n_trials=40
)

[I 2025-11-08 23:13:29,079] A new study created in RDB with name: EASE_R_Recommender_tuning_1


  0%|          | 0/40 [00:00<?, ?it/s]

EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 9.85 sec
[I 2025-11-08 23:13:50,028] Trial 0 finished with value: 0.28263407945632935 and parameters: {'l2': 282.7916012471132, 'topK': 1000}. Best is trial 0 with value: 0.28263407945632935.
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 10.33 sec
[I 2025-11-08 23:14:12,314] Trial 1 finished with value: 0.27114054560661316 and parameters: {'l2': 94.75138785599565, 'topK': 925}. Best is trial 0 with value: 0.28263407945632935.
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 11.43 sec
[I 2025-11-08 23:14:34,876] Trial 2 finished with value: 0.2822282016277313 and parameters: {'l2': 575.8126248533295, 'topK': 900}. Best is trial 0 with value: 0.28263407945632935.
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 10.32 sec
[I 2025-11-08 23:14:56,555] Trial 3 finished with value: 0.28238239884376526 and pa

In [16]:
STUDY_NAME = EASE_R_Recommender.RECOMMENDER_NAME + "_tuning_2"

def tuning_function(optuna_trial: optuna.trial.Trial) -> float:
    recommender_instance = EASE_R_Recommender(URM_train)

    recommender_instance.fit(
        l2_norm=optuna_trial.suggest_float("l2", 200, 600),
        normalize_matrix=False,
        topK=optuna_trial.suggest_int("topK", 700, 850)
    )

    return evaluate_recommender(recommender_instance, at=20)

In [17]:
save_results, optuna_study = hyperparameter_tuning(
    tuning_function,
    study_name=STUDY_NAME,
    n_trials=40
)

[I 2025-11-08 23:31:06,320] A new study created in RDB with name: EASE_R_Recommender_tuning_2


  0%|          | 0/40 [00:00<?, ?it/s]

EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 10.50 sec
[I 2025-11-08 23:31:27,994] Trial 0 finished with value: 0.2819439470767975 and parameters: {'l2': 590.3042116136555, 'topK': 776}. Best is trial 0 with value: 0.2819439470767975.
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 11.34 sec
[I 2025-11-08 23:31:50,297] Trial 1 finished with value: 0.283046156167984 and parameters: {'l2': 387.09356585576336, 'topK': 810}. Best is trial 1 with value: 0.283046156167984.
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 9.98 sec
[I 2025-11-08 23:32:11,402] Trial 2 finished with value: 0.28265678882598877 and parameters: {'l2': 472.234021308291, 'topK': 813}. Best is trial 1 with value: 0.283046156167984.
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 10.05 sec
[I 2025-11-08 23:32:32,277] Trial 3 finished with value: 0.27966809272766113 and parameters

In [18]:
optuna.visualization.plot_optimization_history(optuna_study)

In [19]:
optuna.visualization.plot_param_importances(optuna_study)

In [20]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- Best Value: 0.2832670509815216
- Best Params: {'use_topk': True, 'topK': 850, 'l2': 400.12671834453766, 'normalize': False}